# Task 4 — Notebook 02b: Hyperparameter Tuning (Optuna)

Notebook 02 established the ablation study with hand-set LightGBM hyperparameters.
Here we use **Optuna's TPE sampler** (Bayesian optimisation) to search over a
9-dimensional hyperparameter space and squeeze more accuracy out of the same
feature set. The search is budgeted at ~25 trials / 30 minutes.

**What changes from NB02:**
- Hyperparameters are **tuned**, not hand-picked.
- `rotation_regime` is integer-encoded and included as a feature.
- The best trial is retrained on train+val and evaluated on the 2023 test year.

**Outputs:**
- Tuned model saved to `artifacts/models/task4/crop_type_model_tuned.joblib`
- Updated test predictions, confusion matrix, and before/after comparison table

In [1]:
import json
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root.")
sys.path.insert(0, str(REPO_ROOT))

with open(REPO_ROOT / "configs" / "task4_crop_mapping.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

FIGURES_DIR = REPO_ROOT / Path(cfg["output"]["figures_dir"])
TABLES_DIR  = REPO_ROOT / Path(cfg["output"]["tables_dir"])
MODELS_DIR  = REPO_ROOT / Path(cfg["output"]["models_dir"])
PROCESSED_DIR = REPO_ROOT / Path(cfg["output"]["processed_dir"])
for d in (FIGURES_DIR, TABLES_DIR, MODELS_DIR, PROCESSED_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = cfg["run"]["seed"]
CLASS_NAMES = ["other_cropland", "corn", "soybean", "winter_wheat"]
print(f"REPO_ROOT: {REPO_ROOT}")

REPO_ROOT: C:\Users\sardo\OneDrive\Desktop\Classes\Projects\GeoCrop-Spatiotemporal-Modeling


## 1. Load Panel and Split

In [2]:
from src.preprocessing.task4_panel import train_val_test_split

panel   = pd.read_parquet(PROCESSED_DIR / "feature_matrix_panel.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test_frame_2023.parquet")

train_df, val_df = train_val_test_split(panel, cfg)
train_df = train_df[np.isfinite(train_df["label"])].copy()
val_df   = val_df[np.isfinite(val_df["label"])].copy()
test_df  = test_df[np.isfinite(test_df["label"])].copy()

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")

Train: 4,500,000  |  Val: 500,000  |  Test: 500,000


## 2. Encode `rotation_regime` as an Integer Feature

This column is a categorical string (`irregular`, `monoculture`, `regular`).
Integer-encoding it gives the model a compact summary of each pixel's
historical cropping pattern without adding new data.

In [3]:
REGIME_MAP = {"irregular": 0, "monoculture": 1, "regular": 2}

for df in (train_df, val_df, test_df):
    if "rotation_regime" in df.columns:
        df["rotation_regime_enc"] = df["rotation_regime"].map(REGIME_MAP).astype("Int32")

print("rotation_regime_enc value counts (train):")
print(train_df["rotation_regime_enc"].value_counts().sort_index())

rotation_regime_enc value counts (train):
rotation_regime_enc
0    2628657
1    1107405
2     763938
Name: count, dtype: Int64


## 3. Build Feature List

In [4]:
from src.modeling.crop_type_model import default_feature_columns

feature_cols = default_feature_columns(train_df)
print(f"Features ({len(feature_cols)}): {feature_cols}")

Features (39): ['cdl_t1', 'cdl_t2', 'cdl_t3', 'cdl_t4', 'cdl_t5', 'n_corn_to_soy', 'n_soy_to_corn', 'n_corn_corn', 'n_soy_soy', 'time_since_last_corn', 'time_since_last_soy', 'frac_years_corn', 'frac_years_soy', 'max_run_length', 'alternation_score', 'pattern_distance', 'sequence_entropy', 'neigh_frac_corn', 'neigh_frac_soy', 'ndvi_base', 'ndvi_peak', 'ndvi_amplitude', 'ndvi_mean', 'ndvi_integral', 'ndvi_peak_week', 'ndvi_greenup_week', 'ndvi_greenup_slope', 'ndvi_early_mean', 'ndvi_mid_mean', 'ndvi_late_mean', 'smap_mean_gs', 'smap_spring_sm', 'smap_pct_wet_weeks', 'smap_pct_dry_weeks', 'ndvi_peak_hist_mean', 'ndvi_peak_hist_std', 'ndvi_peak_week_hist_mean', 'ndvi_peak_week_hist_std', 'rotation_regime_enc']


## 4. Baseline: Current Hand-Set Hyperparameters

Quickly retrain with the original config so we have a clean before/after
comparison on the same feature set (which now includes `rotation_regime_enc`).

In [5]:
from src.modeling.crop_type_model import (
    evaluate_multiclass,
    save_model,
    train_lightgbm_classifier,
)

base_hp = dict(cfg["model"]["hyperparameters"])
base_hp.setdefault("objective", "multiclass")
base_hp["num_class"] = len(CLASS_NAMES)
base_hp["n_estimators"] = int(base_hp.get("n_estimators", 500))
base_hp["verbose"] = -1
es = int(cfg["model"].get("early_stopping_rounds", 50))

base_clf = train_lightgbm_classifier(
    train_df, val_df, feature_cols, hp=base_hp, early_stopping_rounds=es,
)
y_val_base = base_clf.predict(val_df[feature_cols])
base_val_metrics = evaluate_multiclass(val_df["label"].values, y_val_base, class_names=CLASS_NAMES)
print(f"Baseline  val OA: {base_val_metrics['overall_accuracy']:.4f}  |  macro F1: {base_val_metrics['macro_f1']:.4f}")

Baseline  val OA: 0.8228  |  macro F1: 0.8242


## 5. Optuna Hyperparameter Search

We use the TPE (Tree-structured Parzen Estimator) sampler — a Bayesian
approach that focuses on promising regions of the search space after the
first few random trials. The objective is **macro F1** on the 2022
validation set, consistent with the ablation study.

In [8]:
from src.modeling.crop_type_model import tune_lightgbm_optuna

tuning_cfg = cfg.get("tuning", {})
n_trials = tuning_cfg.get("n_trials", 25)
timeout  = tuning_cfg.get("timeout", 1800)
ss       = tuning_cfg.get("search_space", {})

print(f"Starting Optuna search: {n_trials} trials, {timeout}s timeout")
print(f"Search space overrides from config: {ss}")
t0 = time.time()

result = tune_lightgbm_optuna(
    train_df, val_df, feature_cols,
    n_trials=n_trials,
    timeout=timeout,
    n_classes=len(CLASS_NAMES),
    seed=SEED,
    search_space=ss,
)

elapsed = time.time() - t0
print(f"\nDone in {elapsed / 60:.1f} min ({len(result['study'].trials)} trials completed)")
print(f"Best macro F1 (val): {result['best_value']:.4f}")
print(f"Best params: {json.dumps(result['best_params'], indent=2)}")

Starting Optuna search: 25 trials, 1800s timeout
Search space overrides from config: {'n_estimators_lo': 500, 'n_estimators_hi': 3000, 'learning_rate_lo': 0.005, 'learning_rate_hi': 0.1, 'max_depth_lo': 5, 'max_depth_hi': 12, 'num_leaves_lo': 31, 'num_leaves_hi': 255, 'min_child_samples_lo': 5, 'min_child_samples_hi': 100, 'subsample_lo': 0.6, 'subsample_hi': 1.0, 'colsample_bytree_lo': 0.5, 'colsample_bytree_hi': 1.0, 'reg_alpha_lo': 0.001, 'reg_alpha_hi': 10.0, 'reg_lambda_lo': 0.001, 'reg_lambda_hi': 10.0}


[W 2026-04-13 12:57:57,821] Trial 1 failed with parameters: {'n_estimators': 2300, 'learning_rate': 0.005318033256270142, 'max_depth': 12, 'num_leaves': 218, 'min_child_samples': 25, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.5917022549267169, 'reg_alpha': 0.016480446427978974, 'reg_lambda': 0.12561043700013558} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\sardo\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\sardo\OneDrive\Desktop\Classes\Projects\GeoCrop-Spatiotemporal-Modeling\src\modeling\crop_type_model.py", line 181, in objective
    clf.fit(
  File "c:\Users\sardo\anaconda3\Lib\site-packages\lightgbm\sklearn.py", line 1560, in fit
    super().fit(
  File "c:\Users\sardo\anaconda3\Lib\site-packages\lightgbm\sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  F

KeyboardInterrupt: 

In [9]:
# After interrupting, the 'study' object may not exist yet.
# Re-import and check if any trials completed:
try:
    print(f"Trials completed: {len(result['study'].trials)}")
    print(f"Best macro F1 so far: {result['best_value']:.4f}")
except NameError:
    print("The result variable was not assigned — the search was interrupted before any trial finished.")

The result variable was not assigned — the search was interrupted before any trial finished.


## 6. Optuna Diagnostics

In [ ]:
study = result["study"]
trials_df = study.trials_dataframe().sort_values("value", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot([t.value for t in study.trials], marker="o", ms=4, alpha=0.7)
axes[0].axhline(result["best_value"], color="green", ls="--", label=f"best = {result['best_value']:.4f}")
axes[0].axhline(base_val_metrics["macro_f1"], color="red", ls="--", label=f"baseline = {base_val_metrics['macro_f1']:.4f}")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("Macro F1 (val)")
axes[0].set_title("Optimisation History")
axes[0].legend(fontsize=8)

importances = {}
for t in study.trials:
    for k, v in t.params.items():
        importances.setdefault(k, []).append((v, t.value))
param_names = list(result["best_params"].keys())
best_vals = [result["best_params"][p] for p in param_names]
axes[1].barh(param_names, best_vals, color="steelblue", alpha=0.8)
axes[1].set_xlabel("Best Value")
axes[1].set_title("Best Trial Parameters")

plt.tight_layout()
stamp = datetime.now(timezone.utc).strftime("%Y%m%d")
fig.savefig(FIGURES_DIR / f"task4_optuna_diagnostics__{stamp}.png", dpi=150, bbox_inches="tight")
plt.show()

display(trials_df.head(10))

## 7. Retrain Best Model on Train+Val → Evaluate on Test (2023)

We take the best trial's hyperparameters, retrain on the combined
train+validation set, and evaluate on the held-out 2023 test year.

In [ ]:
best_hp = {
    "objective": "multiclass",
    "num_class": len(CLASS_NAMES),
    "verbosity": -1,
    "random_state": SEED,
    "is_unbalance": True,
    **result["best_params"],
}

trainval_df = pd.concat([train_df, val_df], ignore_index=True)
split_idx = len(train_df)
final_train = trainval_df.iloc[:split_idx]
final_val   = trainval_df.iloc[split_idx:]

print(f"Retraining with best HP on train+val ({len(trainval_df):,} rows) ...")

tuned_clf = train_lightgbm_classifier(
    final_train, final_val, feature_cols,
    hp=best_hp, early_stopping_rounds=100,
)

y_test_pred = tuned_clf.predict(test_df[feature_cols])
tuned_metrics = evaluate_multiclass(test_df["label"].values, y_test_pred, class_names=CLASS_NAMES)

print(f"\nTuned Model — Test-Year ({cfg['panel']['test_year']}) Results:")
print(f"  Overall accuracy: {tuned_metrics['overall_accuracy']:.4f}")
print(f"  Macro F1:         {tuned_metrics['macro_f1']:.4f}")
for cname in CLASS_NAMES:
    print(f"  F1 {cname}: {tuned_metrics['per_class_f1_named'].get(cname, 0.0):.4f}")

## 8. Before / After Comparison

In [ ]:
baseline_test_path = TABLES_DIR / sorted(
    TABLES_DIR.glob("task4__test_metrics__*.json"), key=lambda p: p.name
)[-1].name if list(TABLES_DIR.glob("task4__test_metrics__*.json")) else None

if baseline_test_path and baseline_test_path.is_file():
    old_metrics = json.loads(baseline_test_path.read_text(encoding="utf-8"))
else:
    old_metrics = {"overall_accuracy": 0.7920, "macro_f1": 0.7910,
                   "per_class_f1_named": {"other_cropland": 0.886, "corn": 0.720,
                                          "soybean": 0.735, "winter_wheat": 0.813}}
    print("(Using fallback baseline numbers from NB02.)")

rows = []
for label, m in [("Baseline (NB02)", old_metrics), ("Tuned (NB02b)", tuned_metrics)]:
    row = {"model": label,
           "OA": m["overall_accuracy"],
           "macro_F1": m["macro_f1"]}
    for cn in CLASS_NAMES:
        row[f"F1_{cn}"] = m.get("per_class_f1_named", {}).get(cn, 0.0)
    rows.append(row)

compare_df = pd.DataFrame(rows)
delta_row = {"model": "Δ (tuned − baseline)"}
for col in compare_df.columns[1:]:
    delta_row[col] = compare_df.iloc[1][col] - compare_df.iloc[0][col]
compare_df = pd.concat([compare_df, pd.DataFrame([delta_row])], ignore_index=True)

display(compare_df.style.format({c: "{:.4f}" for c in compare_df.columns[1:]}))

compare_csv = TABLES_DIR / f"task4__tuning_comparison__{stamp}.csv"
compare_df.to_csv(compare_csv, index=False)
print(f"Wrote {compare_csv}")

## 9. Save Tuned Model and Predictions

In [ ]:
tuned_model_path = MODELS_DIR / "crop_type_model_tuned.joblib"
save_model(tuned_clf, tuned_model_path)
print(f"Saved tuned model → {tuned_model_path}")

tuned_metrics_path = TABLES_DIR / f"task4__test_metrics_tuned__{stamp}.json"
tuned_metrics_path.write_text(json.dumps(tuned_metrics, indent=2), encoding="utf-8")
print(f"Saved metrics   → {tuned_metrics_path}")

best_params_path = TABLES_DIR / f"task4__best_hparams__{stamp}.json"
best_params_path.write_text(json.dumps(result["best_params"], indent=2), encoding="utf-8")
print(f"Saved best HP   → {best_params_path}")

pred_df = test_df[["iy", "ix", "label", "year"]].copy()
if "rotation_regime" in test_df.columns:
    pred_df["rotation_regime"] = test_df["rotation_regime"]
pred_df["predicted"] = y_test_pred.astype(np.float32)
pred_path = PROCESSED_DIR / "test_predictions_2023_tuned.parquet"
pred_df.to_parquet(pred_path, index=False, compression="zstd")
print(f"Saved predictions → {pred_path}")

## 10. Confusion Matrix (Tuned Model)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_df["label"].astype(int).values, y_test_pred.astype(int))
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm_pct, annot=True, fmt=".2%", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Tuned Model — Test {cfg['panel']['test_year']} Confusion Matrix")
plt.tight_layout()
fig.savefig(FIGURES_DIR / f"task4_test_confusion_matrix_tuned__{stamp}.png", dpi=150, bbox_inches="tight")
plt.show()

print("Raw counts:")
print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

## 11. SHAP Feature Importance (Tuned Model)

In [ ]:
import shap

shap_n = cfg.get("feature_importance", {}).get("shap_sample_size", 1000)
idx = np.random.RandomState(SEED).choice(len(test_df), size=min(shap_n, len(test_df)), replace=False)
X_shap = test_df[feature_cols].iloc[idx]

explainer = shap.TreeExplainer(tuned_clf)
shap_values = explainer.shap_values(X_shap)

if isinstance(shap_values, list):
    mean_abs = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
else:
    mean_abs = np.abs(shap_values).mean(axis=0)
    if mean_abs.ndim == 2:
        mean_abs = mean_abs.mean(axis=1)

imp_df = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs})
imp_df = imp_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 8))
top20 = imp_df.head(20)
ax.barh(top20["feature"][::-1], top20["mean_abs_shap"][::-1], color="steelblue")
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("Top-20 Feature Importance (Tuned Model)")
plt.tight_layout()
fig.savefig(FIGURES_DIR / f"task4_shap_importance_tuned__{stamp}.png", dpi=150, bbox_inches="tight")
plt.show()

imp_csv = TABLES_DIR / f"task4_shap_importance_tuned__{stamp}.csv"
imp_df.to_csv(imp_csv, index=False)
print(f"Wrote {imp_csv}")

## 12. Spatial Prediction Maps (Tuned Model)

In [ ]:
from src.viz.prediction_maps import labels_to_raster, plot_crop_type_map

meta_path = REPO_ROOT / cfg["cdl"]["metadata_path"]
with open(meta_path.with_suffix(".json"), encoding="utf-8") as f:
    meta = json.load(f)
HEIGHT, WIDTH = meta["height"], meta["width"]
NODATA = -1

pred_raster = labels_to_raster(
    pred_df["iy"].values, pred_df["ix"].values,
    pred_df["predicted"].values.astype(int),
    HEIGHT, WIDTH, nodata=NODATA,
)
true_raster = labels_to_raster(
    pred_df["iy"].values, pred_df["ix"].values,
    pred_df["label"].values.astype(int),
    HEIGHT, WIDTH, nodata=NODATA,
)

CLASS_COLORS = ["#bdbdbd", "#ffd700", "#228b22", "#d2691e"]

try:
    from src.viz.rotation_maps import load_cornbelt_state_boundaries_5070
    states = load_cornbelt_state_boundaries_5070()
except Exception:
    states = None

t = meta.get("transform")
if t:
    from affine import Affine
    aff = Affine(*t[:6]) if isinstance(t, list) else Affine(t["a"], t["b"], t["c"], t["d"], t["e"], t["f"])
    left, top = aff * (0, 0)
    right, bottom = aff * (WIDTH, HEIGHT)
    geo_extent = (left, right, bottom, top)
else:
    geo_extent = None

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
plot_crop_type_map(
    true_raster, ax=axes[0],
    class_names=CLASS_NAMES, class_colors=CLASS_COLORS,
    title=f"True Crop Types (CDL {cfg['panel']['test_year']})",
    nodata=NODATA, extent=geo_extent, state_shapes=states,
)
plot_crop_type_map(
    pred_raster, ax=axes[1],
    class_names=CLASS_NAMES, class_colors=CLASS_COLORS,
    title=f"Tuned Predicted Crop Types ({cfg['panel']['test_year']})",
    nodata=NODATA, extent=geo_extent, state_shapes=states,
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / f"task4_crop_maps_tuned__{stamp}.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

- **Baseline (NB02):** Hand-set hyperparameters, 38 features
- **Tuned (NB02b):** Optuna TPE search over 9 dimensions + `rotation_regime_enc` feature
- Comparison table and all artifacts saved above.

If the tuned model shows meaningful improvement, use
`crop_type_model_tuned.joblib` and the `*_tuned` predictions in
downstream notebooks (NB03, NB04).